# jex: complete configured Colab GPU run

Select a GPU runtime, then **Run all**. Setup may remove an unused old TorchAO package that conflicts with pinned PEFT; `COLAB_OPTIONAL_PACKAGE_CLEANUP` explicitly enables this only after hosted Colab runtime checks. Core package versions are preserved and the change is logged. This notebook uses public, pinned clones, with no GitHub token, Google Drive mount, or paid model API. Runtime storage is temporary: download both output ZIPs before disconnecting.

Scope matches the original `jex_gpu.ipynb`: Qwen2.5-1.5B student, 4-bit Qwen2.5-7B teacher; **400/50/150/400** train/dev/eval/held-out records per task; **five head variants**; **3,000-record LoRA**; **four benchmark models at 30 examples per dataset**; latency K=1/2/4/8/16 with `--repeats 5`; async rates 5/20/50 with 200 arrivals each. No automatic downscaling. The 30-example benchmark is the original configured sample, **not the full benchmark dataset evaluation**. A full-dataset extension is separate and is not launched here.

Subprocess failures are visible and logged, and block dependent stages. Independent stages continue. The final status is incomplete if a stage failed or was blocked. Rerun in the same runtime to retain completed stages and retry unsuccessful ones. Package versions, source commits, commands, statuses, timings, coverage, and artifact hashes are retained. Exact training/evaluation records are saved; unpinned upstream Hub revisions and GPU nondeterminism still limit reproducibility.

**Resources:** LoRA retains the original batch-four and dtype selection, without gradient checkpointing. The original code uses `torch.cuda.is_bf16_supported()`; newer PyTorch versions can report emulated BF16 support on T4 and select BF16 despite the original comment saying FP32. Memory exhaustion is still possible; this runner reports failure rather than shrinking the experiment. Total runtime and resource usage depend on the GPU and have not been established by notebook preparation. Dataset gates and network failures are recorded in logs and coverage.

**Review the released response-data license before enabling Jev comparison.** `JEV_RESPONSE_LICENSE_ACK` defaults to `False`, so Jev download/scoring/comparison remain explicitly blocked while independent work runs. Set it to `True` only if the intended use complies with [the response-data license](https://github.com/AppliedMachineLearning-Lab/jev-benchmarking/blob/6bbdeb33474849b6de2f0cccc9f5e19756abd67e/responses/LICENSE_RESPONSES.md). The acknowledgment does not waive restrictions. Jev responses are used only for research comparison, never for training, distillation, targets, or optimization. Their license also prohibits developing or facilitating a similar or competing product. Qwen3.8 and Gemma responses use Apache 2.0. Exports include unchanged licenses and attribution; source response databases are excluded.

Reference: Deußer, Sparrenberg, Sifa, *Evaluating and Benchmarking the System One Model Jev*, arXiv:2609.37647, dataset DOI [10.5281/zenodo.23039006](https://doi.org/10.5281/zenodo.23039006).

Pinned jex: `c494049aa3af480abc5ec502a10a783b26c2a71b`. Pinned harness: `6bbdeb33474849b6de2f0cccc9f5e19756abd67e`. See `docs/COLAB_FULL_RUN.md` for outputs, caveats, and recovery.


In [ ]:
from pathlib import Path
import hashlib

RUN_ROOT = Path("/content/jex-full-run")
# Review the response-data license and intended use before enabling.
JEV_RESPONSE_LICENSE_ACK = False
# Only in verified hosted Colab: remove unused TorchAO <0.16.0 if incompatible with pinned PEFT.
COLAB_OPTIONAL_PACKAGE_CLEANUP = True
RUNNER = Path("/content/colab_full_run.py")
RUNNER_SOURCE = '#!/usr/bin/env python3\n"""Full, resumable GPU run of the pinned jex notebook. No credentials or Drive mount.\n\nOnly stdlib is imported in the orchestration process. Training, tests, evaluation,\nand GPU benchmarks run in fresh subprocesses after dependencies are installed.\nRun: python scripts/colab_full_run.py --root /content/jex-full-run\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nfrom importlib import metadata as package_metadata, util as import_util\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport shlex\nimport shutil\nimport sqlite3\nimport subprocess\nimport sys\nimport time\nimport traceback\nimport urllib.request\nimport zipfile\n\nJEX_URL = "https://github.com/dffdeeq/jex.git"\nJEX_COMMIT = "c494049aa3af480abc5ec502a10a783b26c2a71b"\nHARNESS_URL = "https://github.com/AppliedMachineLearning-Lab/jev-benchmarking.git"\nHARNESS_COMMIT = "6bbdeb33474849b6de2f0cccc9f5e19756abd67e"\nLICENSE_URL = f"https://github.com/AppliedMachineLearning-Lab/jev-benchmarking/blob/{HARNESS_COMMIT}/responses/LICENSE_RESPONSES.md"\nATTRIBUTION = """Benchmark code and released response-data reference:\nTobias Deußer, Lorenz Sparrenberg and Rafet Sifa (2026).\nEvaluating and Benchmarking the System One Model Jev. arXiv:2609.37647.\nPaper: https://arxiv.org/abs/2609.37647\nResponse dataset: https://doi.org/10.5281/zenodo.23039006\n\nJev response data is for licensed research/evaluation use only; no training,\ndistillation, optimization, or competing-product development. Open-model\nresponse data is separately licensed under Apache 2.0. Consult the unchanged\nresponse-data license included with results when the harness is available.\nSource response databases are not included in this archive.\n"""\nSTUDENT = "Qwen/Qwen2.5-1.5B-Instruct"\nTEACHER = "Qwen/Qwen2.5-7B-Instruct"\nSIZES = {"train_per_task": 400, "dev_per_task": 50, "eval_per_task": 150, "heldout_per_task": 400}\nHEADS = {\n    "jex-head": [],\n    "ablation-gold-only": ["--no-teacher"],\n    "ablation-no-rlcd": ["--rl-epochs", "0"],\n    "ablation-no-memory": ["--no-memory"],\n    "prior-kl-1.0": ["--no-memory", "--prior-kl", "1.0"],\n}\nBENCH_MODELS = {\n    "qwen2.5-1.5b-zs": (STUDENT, None),\n    "jex-lora": ("artifacts/lora", None),\n    "qwen3.5-4b-zs": ("Qwen/Qwen3.5-4B", None),\n    "qwen3.5-9b-zs": ("Qwen/Qwen3.5-9B", "4bit"),\n}\nBASELINES = {\n    "Qwen3.8-27B": ("hf:Qwen/Qwen3.8-27B", "responses.Qwen__Qwen3.8-27B.db"),\n    "Gemma-4-E4B": ("hf:google/gemma-4-E4B-it", "responses.google__gemma-4-E4B-it.db"),\n    "Jev": ("jev-1.13.0", "responses.db"),\n}\nBENCH_LIMIT = 30\n# Preserve the Colab CUDA PyTorch build. Other resolved versions are captured by\n# pip freeze and the environment manifest, rather than upgrading torch blindly.\nPINNED_EXTRA_PACKAGES = ["transformers==5.14.1", "peft==0.21.2", "flash-linear-attention==0.5.2"]\nCONFIG = {\n    "jex_commit": JEX_COMMIT, "harness_commit": HARNESS_COMMIT,\n    "student": STUDENT, "teacher": TEACHER, "teacher_quant": "4bit",\n    "sizes": SIZES, "heads": HEADS, "lora_max_records": 3000,\n    "lora_epochs": 1, "lora_batch_records": 4,\n    "bench_models": BENCH_MODELS, "bench_limit": BENCH_LIMIT,\n    "latency_ks": [1, 2, 4, 8, 16], "latency_repeats": 5,\n    "async_rates": [5, 20, 50], "async_n": 200, "seed": 0,\n    "pinned_extra_packages": PINNED_EXTRA_PACKAGES,\n}\n\n\ndef utc():\n    return time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + ".tmp")\n    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\\n")\n    temporary.replace(path)\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef installed_version(name):\n    try:\n        return package_metadata.version(name)\n    except package_metadata.PackageNotFoundError:\n        return None\n\n\ndef torchao_is_incompatible(version):\n    """Conservatively classify stable release metadata against PEFT 0.21.2.\n\n    Prerelease or unrecognized versions are not grounds for automatic removal.\n    No ML package needs to be imported for this check.\n    """\n    match = re.fullmatch(r"(\\d+)\\.(\\d+)(?:\\.(\\d+))?(?:\\+[A-Za-z0-9.-]+)?", version)\n    if match is None:\n        return None\n    return tuple(int(value or 0) for value in match.groups()) < (0, 16, 0)\n\n\nclass Blocked(RuntimeError):\n    pass\n\n\nclass Runner:\n    def __init__(self, root, *, jev_license_ack=False, allow_colab_optional_package_cleanup=False):\n        self.root = Path(root).resolve()\n        self.repo = self.root / "jex"\n        self.harness = self.root / "jev-benchmarking"\n        self.logs = self.root / "logs"\n        self.logs.mkdir(parents=True, exist_ok=True)\n        self.jev_license_ack = jev_license_ack\n        self.allow_colab_optional_package_cleanup = allow_colab_optional_package_cleanup\n        self.status_path = self.root / "stage_status.json"\n        self.status = json.loads(self.status_path.read_text()) if self.status_path.exists() else {}\n        config_path = self.root / "run_config.json"\n        expected = json.loads(json.dumps(CONFIG))\n        if config_path.exists() and json.loads(config_path.read_text()) != expected:\n            raise RuntimeError("Existing run configuration differs. Use a NEW root directory; do not mix runs.")\n        write_json(config_path, CONFIG)\n        (self.root / "ATTRIBUTION.txt").write_text(ATTRIBUTION)\n        self.env = dict(os.environ, PYTHONUNBUFFERED="1", PYTHONHASHSEED="0",\n                        TOKENIZERS_PARALLELISM="false", HF_HUB_DISABLE_TELEMETRY="1",\n                        GIT_TERMINAL_PROMPT="0")\n        # These are offline/cached evaluations, not live TypeSafe API calls.\n        self.env.pop("TYPESAFE_API_KEY", None)\n        self.env.pop("TYPESAFE_API_KEY_FILE", None)\n        self.current = None\n        self.logfile = None\n        self.plan = []\n\n    def emit(self, line):\n        print(line, end="" if line.endswith("\\n") else "\\n", flush=True)\n        if self.logfile:\n            self.logfile.write(line if line.endswith("\\n") else line + "\\n")\n            self.logfile.flush()\n\n    def command(self, args, *, cwd=None, stdout_file=None):\n        args = [str(a) for a in args]\n        self.emit("$ " + shlex.join(args))\n        proc = subprocess.Popen(args, cwd=cwd or self.repo, env=self.env,\n                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\n                                text=True, encoding="utf-8", errors="replace", bufsize=1)\n        capture = Path(stdout_file).open("w") if stdout_file else None\n        try:\n            assert proc.stdout is not None\n            for line in proc.stdout:\n                self.emit(line)\n                if capture:\n                    capture.write(line)\n            code = proc.wait()\n            if code:\n                raise subprocess.CalledProcessError(code, args)\n        except BaseException:\n            if proc.poll() is None:\n                proc.terminate()\n                try:\n                    proc.wait(timeout=10)\n                except subprocess.TimeoutExpired:\n                    proc.kill()\n                    proc.wait()\n            raise\n        finally:\n            if proc.stdout is not None:\n                proc.stdout.close()\n            if capture:\n                capture.close()\n\n    def output(self, args, cwd=None):\n        return subprocess.check_output(args, cwd=cwd or self.repo, env=self.env, text=True).strip()\n\n    def python(self, script, *args, cwd=None):\n        self.command([sys.executable, "-u", self.root / "seeded_run.py", script, *args], cwd=cwd)\n\n    def clone(self, url, commit, destination):\n        if not destination.exists():\n            self.command(["git", "clone", "--no-checkout", url, destination], cwd=self.root)\n        actual_url = self.output(["git", "remote", "get-url", "origin"], cwd=destination)\n        if actual_url != url:\n            raise RuntimeError(f"Refusing unrelated existing checkout: {destination}")\n        if not (destination / ".git").exists():\n            raise RuntimeError(f"Not a git repository: {destination}")\n        self.command(["git", "checkout", "--detach", commit], cwd=destination)\n        actual = self.output(["git", "rev-parse", "HEAD"], cwd=destination)\n        if actual != commit:\n            raise RuntimeError(f"Revision mismatch: {actual}")\n        self.emit(f"Verified {url} @ {actual}")\n\n    def add(self, name, fn, deps=(), outputs=(), *, always=False):\n        self.plan.append((name, fn, tuple(deps), [Path(p) for p in outputs], always))\n\n    def stage(self, name, fn, deps, outputs, always):\n        blockers = [d for d in deps if self.status.get(d, {}).get("status") != "succeeded"]\n        if blockers:\n            self.status[name] = {"status": "blocked", "reason": "Dependencies not succeeded: " + ", ".join(blockers), "updated": utc()}\n            write_json(self.status_path, self.status)\n            self.emit(f"BLOCKED {name}: {\', \'.join(blockers)}")\n            return\n        previous = self.status.get(name, {})\n        if (not always and outputs and previous.get("status") == "succeeded"\n                and all(p.is_file() and p.stat().st_size > 0 for p in outputs)):\n            self.emit(f"RESUME {name}: completed stage retained")\n            return\n        row = {"status": "running", "started": utc(), "attempt": previous.get("attempt", 0) + 1,\n               "outputs": [str(p.relative_to(self.root)) for p in outputs]}\n        self.status[name] = row\n        write_json(self.status_path, self.status)\n        t0 = time.monotonic()\n        self.current = name\n        with (self.logs / f"{name}.log").open("a") as log:\n            self.logfile = log\n            self.emit(f"\\n===== {name} [{utc()}] =====")\n            try:\n                fn()\n                missing = [str(p) for p in outputs if not p.is_file() or p.stat().st_size == 0]\n                if missing:\n                    raise RuntimeError("Missing/empty expected outputs: " + ", ".join(missing))\n                row["status"] = "succeeded"\n            except Blocked as exc:\n                row.update(status="blocked", reason=str(exc))\n                self.emit(f"BLOCKED: {exc}")\n            except Exception as exc:\n                row.update(status="failed", error=f"{type(exc).__name__}: {exc}")\n                self.emit(traceback.format_exc())\n            except BaseException:\n                row.update(status="interrupted", error="Notebook interrupted; rerun to resume")\n                raise\n            finally:\n                row.update(finished=utc(), seconds=time.monotonic() - t0)\n                write_json(self.status_path, self.status)\n                self.logfile = None\n                self.current = None\n                self.bundle()\n        self.emit(f"{row[\'status\'].upper()} {name} ({row[\'seconds\']:.1f}s)")\n\n    def colab_runtime_evidence(self):\n        try:\n            has_colab_package = import_util.find_spec("google.colab") is not None\n        except (ImportError, ValueError):\n            has_colab_package = False\n        return {\n            "colab_package_available": has_colab_package,\n            "colab_release_tag_present": bool(os.environ.get("COLAB_RELEASE_TAG")),\n            "content_directory_present": Path("/content").is_dir(),\n        }\n\n    def prepare_optional_backends(self):\n        """Remove only an unused incompatible TorchAO, with explicit Colab opt-in."""\n        names = ("torch", "peft", "transformers", "torchao")\n        before = {name: installed_version(name) for name in names}\n        if before["torchao"] is None:\n            self.emit("Optional TorchAO is absent; no compatibility cleanup needed.")\n            return\n        if before["peft"] != "0.21.2":\n            raise RuntimeError("Unexpected PEFT version; refusing automatic optional-backend changes")\n        incompatible = torchao_is_incompatible(before["torchao"])\n        if incompatible is False:\n            self.emit(f"Optional TorchAO {before[\'torchao\']} meets PEFT 0.21.2\'s >=0.16.0 requirement; unchanged.")\n            return\n        evidence = self.colab_runtime_evidence()\n        reason = (f"Unused TorchAO {before[\'torchao\']} is incompatible with pinned PEFT 0.21.2 "\n                  "(requires >=0.16.0 when TorchAO is installed). jex uses ordinary LoRA and bitsandbytes, not TorchAO.")\n        if incompatible is None:\n            reason = f"Cannot safely classify installed TorchAO {before[\'torchao\']!r} against PEFT 0.21.2\'s >=0.16.0 requirement."\n        record = {"checked_at": utc(), "before": before, "reason": reason,\n                  "cleanup_explicitly_enabled": self.allow_colab_optional_package_cleanup,\n                  "runtime_evidence": evidence, "status": "blocked"}\n        path = self.root / "optional_backend_cleanup.json"\n        write_json(path, record)\n        self.emit("Optional-backend check: " + json.dumps(record, sort_keys=True))\n        remedy = ("No package was removed. Use a fresh isolated virtual environment without --system-site-packages "\n                  "and install the pinned dependencies there. In a hosted ephemeral Colab runtime only, "\n                  "review and enable --allow-colab-optional-package-cleanup.")\n        if incompatible is None:\n            raise RuntimeError(f"Cannot safely classify TorchAO version {before[\'torchao\']!r}. " + remedy)\n        if not self.allow_colab_optional_package_cleanup or not all(evidence.values()):\n            raise RuntimeError(reason + " Automatic removal requires explicit opt-in and verified hosted Colab indicators. " + remedy)\n        self.emit(reason + " Removing only this unused optional package from the ephemeral Colab runtime.")\n        record["status"] = "removing"\n        write_json(path, record)\n        try:\n            self.command([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], cwd=self.root)\n            after = {name: installed_version(name) for name in names}\n            record["after"] = after\n            if after["torchao"] is not None or any(after[name] != before[name] for name in names[:-1]):\n                raise RuntimeError("Optional-backend cleanup verification failed; core package versions must remain unchanged")\n            record["status"] = "removed"\n        except BaseException as exc:\n            record.update(status="failed", error=f"{type(exc).__name__}: {exc}")\n            raise\n        finally:\n            write_json(path, record)\n        self.emit("Verified optional TorchAO removal; torch, PEFT and Transformers versions are unchanged.")\n\n    def setup(self):\n        self.clone(JEX_URL, JEX_COMMIT, self.repo)\n        self.clone(HARNESS_URL, HARNESS_COMMIT, self.harness)\n        self.command([sys.executable, "-m", "pip", "install", "-e", f"{self.repo}[server,dev,gpu]",\n                      "-e", str(self.harness), *PINNED_EXTRA_PACKAGES[:2]], cwd=self.root)\n        self.prepare_optional_backends()\n        (self.repo / "artifacts").mkdir(exist_ok=True)\n        # The upstream harness ships published full-sample results. Preserve them\n        # elsewhere, then score into an empty directory to avoid stale comparisons.\n        results = self.harness / "results"\n        archive = self.harness / "published-results-reference-only"\n        if results.exists() and not archive.exists():\n            results.rename(archive)\n        results.mkdir(exist_ok=True)\n        (self.harness / "cache").mkdir(exist_ok=True)\n        (self.root / "seeded_run.py").write_text(\'\'\'import random, runpy, sys\\nimport numpy as np\\nimport torch\\nrandom.seed(0)\\nnp.random.seed(0)\\ntorch.manual_seed(0)\\nif torch.cuda.is_available(): torch.cuda.manual_seed_all(0)\\nscript = sys.argv[1]\\nsys.argv = sys.argv[1:]\\nsys.path.insert(0, str(__import__(\'pathlib\').Path(script).resolve().parent))\\nrunpy.run_path(script, run_name="__main__")\\n\'\'\')\n        self.command([sys.executable, "-m", "pip", "freeze"], cwd=self.root, stdout_file=self.root / "pip-freeze.txt")\n        self.command(["nvidia-smi"], cwd=self.root, stdout_file=self.root / "nvidia-smi.txt")\n        self.command([sys.executable, "-c", "import torch, peft, transformers, datasets, bitsandbytes, accelerate; "\n            "assert torch.cuda.is_available(), \'GPU required; no CPU experiment fallback\'; "\n            "print(\'torch\',torch.__version__,\'cuda\',torch.version.cuda,\'transformers\',transformers.__version__,"\n            "\'peft\',peft.__version__,\'datasets\',datasets.__version__); "\n            "print(\'GPU\',torch.cuda.get_device_name(0),\'VRAM\',torch.cuda.get_device_properties(0).total_memory,\'bf16\',torch.cuda.is_bf16_supported())"],\n            cwd=self.root, stdout_file=self.root / "environment.txt")\n        shutil.copy2(Path(__file__), self.root / "runner-source.py")\n        self.emit("LoRA retains upstream dtype/batch defaults. PyTorch may report emulated BF16 support on T4; memory use remains unverified. No automatic sample/batch reduction.")\n\n    def download_baseline(self, label):\n        model, filename = BASELINES[label]\n        if label == "Jev" and not self.jev_license_ack:\n            raise Blocked("Jev response-data license review is required before use: " + LICENSE_URL)\n        path = self.harness / "cache" / filename\n        if not path.exists():\n            url = f"https://zenodo.org/api/records/23039006/files/{filename}/content"\n            partial = path.with_suffix(".part")\n            self.emit("Downloading " + url)\n            with urllib.request.urlopen(url, timeout=120) as response, partial.open("wb") as target:\n                shutil.copyfileobj(response, target, length=8 * 1024 * 1024)\n            partial.replace(path)\n        with path.open("rb") as stream:\n            if stream.read(16) != b"SQLite format 3\\x00":\n                raise RuntimeError(f"Not a SQLite response database: {path}")\n        with sqlite3.connect(f"file:{path}?mode=ro", uri=True) as conn:\n            count = conn.execute("SELECT COUNT(*) FROM responses WHERE model=?", (model,)).fetchone()[0]\n            if count == 0:\n                raise RuntimeError(f"Database has no responses for {model}")\n        write_json(self.root / f"baseline-{label}.json", {"url": f"https://zenodo.org/api/records/23039006/files/{filename}/content",\n            "filename": filename, "model": model, "responses": count, "sha256": sha256(path), "license": LICENSE_URL})\n\n    def results_dir(self, model):\n        base = self.harness / "results" / "eval"\n        return base if model == "jev-1.13.0" else base / "open_models" / model.removeprefix("hf:").replace("/", "__")\n\n    def score(self, model):\n        directory = self.results_dir(model)\n        directory.mkdir(parents=True, exist_ok=True)\n        # A partial/failed earlier attempt must not leave stale per-task metrics.\n        for old in directory.glob("*.json"):\n            old.unlink()\n        summary = directory / "summary.md"\n        if summary.exists():\n            summary.unlink()\n        self.python("scripts/evaluate.py", "all", "--split", "eval", "--limit", BENCH_LIMIT,\n                    "--no-ci", "--model", model, cwd=self.harness)\n        rows = []\n        for path in sorted(directory.glob("*.json")):\n            item = json.loads(path.read_text())\n            if isinstance(item, dict) and "n_examples" in item:\n                rows.append({k: item.get(k) for k in ["task", "n_examples", "n_answered", "models", "primary", "skipped_configs"]})\n        if not rows:\n            raise RuntimeError(f"No benchmark tasks scored for {model}; consult logs, do not treat this as success")\n        write_json(self.root / "coverage" / (model.replace("/", "__").replace(":", "_") + ".json"),\n                   {"model": model, "requested_limit": BENCH_LIMIT, "scored_tasks": len(rows), "tasks": rows,\n                    "caveat": "Dataset/config load failures and schema rejections are preserved in logs. A subprocess exit 0 is not full dataset coverage."})\n\n    def compare(self, include_jev):\n        models = (["Jev=jev-1.13.0"] if include_jev else []) + [f"{k}={BASELINES[k][0]}" for k in ("Qwen3.8-27B", "Gemma-4-E4B")]\n        models += [f"{tag}=hf:jex/{tag}" for tag in BENCH_MODELS]\n        # Upstream compare_jev divides by zero on no common tasks. Fail clearly.\n        tables = []\n        for item in models:\n            _, model = item.split("=", 1)\n            tasks = set()\n            for file in self.results_dir(model).glob("*.json"):\n                data = json.loads(file.read_text())\n                if isinstance(data, dict) and (data.get("primary") or {}).get("value") is not None:\n                    tasks.add(file.stem)\n            tables.append(tasks)\n        common = set.intersection(*tables)\n        if not common:\n            raise RuntimeError("No common scored benchmark tasks; cannot compute a comparison mean")\n        if not (common - {"sst2", "ag_news", "emotion", "sst5"}):\n            raise RuntimeError("No common untrained tasks; upstream comparison would divide by zero")\n        args = ["scripts/compare_jev.py", "--harness", str(self.harness)]\n        for item in models:\n            args.extend(["--model", item])\n        args.extend(["--out", "artifacts/jev_compare.md" if include_jev else "artifacts/open_models_compare.md"])\n        self.python(*args)\n        write_json(self.root / ("comparison_coverage.json" if include_jev else "open_comparison_coverage.json"),\n                   {"limit": BENCH_LIMIT, "common_tasks": sorted(common),\n                    "models": {name: {"scored_tasks": sorted(tasks), "excluded_from_common": sorted(tasks - common)} for name, tasks in zip(models, tables)},\n                    "warning": "Aggregate uses common tasks only. Consult each model coverage file and per-task n_answered before interpreting it."})\n\n    def make_plan(self):\n        art = self.repo / "artifacts"\n        self.add("setup", self.setup, always=True)\n        self.add("tests", lambda: self.command([sys.executable, "-m", "pytest", "-q", "--junitxml", str(self.root / "tests.xml")]),\n                 ("setup",), [self.root / "tests.xml"])\n        sizes = [part for key, value in SIZES.items() for part in ["--" + key.replace("_", "-"), str(value)]]\n        def extract():\n            # Keep the exact original records across a same-runtime retry.\n            saved = art / "feats" / "records.pt"\n            resume = ["--records-from", str(saved)] if saved.exists() else []\n            self.python("scripts/extract.py", "--out", "artifacts/feats", "--student", STUDENT,\n                        "--teacher", TEACHER, "--teacher-quant", "4bit", "--device", "cuda", *sizes, *resume)\n        self.add("extract", extract, ("tests",),\n                 [art / "feats" / f"{kind}_{split}.pt" for kind in ["student", "teacher"] for split in ["train", "dev", "eval"]]\n                 + [art / "feats" / "records.pt", art / "feats" / "meta.json"])\n        for name, flags in HEADS.items():\n            self.add("head-" + name, lambda name=name, flags=flags: self.python("scripts/train_head.py", "--out", f"artifacts/{name}", "--device", "cuda", *flags),\n                     ("extract",), [art / name / "head.pt", art / name / "train_info.json", art / name / "jex.json"])\n        self.add("evaluate-heads", lambda: self.python("scripts/evaluate.py", "--heads", *[f"artifacts/{name}" for name in HEADS]),\n                 ["head-" + name for name in HEADS], [art / "eval.json"])\n        self.add("train-lora", lambda: self.python("scripts/train_lora.py", "--out", "artifacts/lora", "--max-records", "3000", "--device", "cuda"),\n                 ("extract",), [art / "lora" / f for f in ["adapter_model.safetensors", "adapter_config.json", "jex.json", "train_info.json"]])\n        self.add("extract-lora", lambda: self.python("scripts/extract.py", "--student", STUDENT, "--student-adapter", "artifacts/lora", "--records-from", "artifacts/feats/records.pt",\n                 "--splits", "dev,eval", "--skip-teacher", "--out", "artifacts/feats-lora", "--device", "cuda"), ("train-lora",),\n                 [art / "feats-lora" / f for f in ["student_dev.pt", "student_eval.pt", "records.pt", "meta.json"]])\n        self.add("evaluate-lora", lambda: self.python("scripts/evaluate.py", "--feats", "artifacts/feats-lora", "--heads", "--out", "artifacts/eval_lora.json"),\n                 ("extract-lora",), [art / "eval_lora.json"])\n        def install_fla():\n            # Tiny Qwen3.5 pytest fixtures live on CPU. Install GPU FLA only after\n            # those tests, matching the original notebook\'s ordering.\n            self.command([sys.executable, "-m", "pip", "install", PINNED_EXTRA_PACKAGES[2]], cwd=self.root)\n            self.command([sys.executable, "-m", "pip", "freeze"], cwd=self.root, stdout_file=self.root / "pip-freeze.txt")\n            (self.root / "fla-installed.txt").write_text(PINNED_EXTRA_PACKAGES[2] + "\\n")\n        self.add("install-fla", install_fla, ("tests",), [self.root / "fla-installed.txt"])\n        for tag, (model, quant) in BENCH_MODELS.items():\n            deps = ("tests", "train-lora") if tag == "jex-lora" else ("tests",)\n            if tag.startswith("qwen3.5"):\n                deps += ("install-fla",)\n            args = ["scripts/jev_bench.py", "--checkpoint" if tag == "jex-lora" else "--backbone", model,\n                    "--tag", tag, "--limit", str(BENCH_LIMIT), "--out", f"runs/{tag}", "--device", "cuda"]\n            if quant:\n                args += ["--quant", quant]\n            self.add("bench-" + tag, lambda args=args: self.python(*args), deps, [self.repo / "runs" / tag / "responses.jsonl"])\n            def import_and_score(tag=tag):\n                self.python("scripts/import_responses.py", str(self.repo / "runs" / tag / "responses.jsonl"), cwd=self.harness)\n                self.score("hf:jex/" + tag)\n            self.add("score-" + tag, import_and_score, ("bench-" + tag,), [self.results_dir("hf:jex/" + tag) / "summary.md"])\n        for label, (model, _) in BASELINES.items():\n            self.add("download-" + label, lambda label=label: self.download_baseline(label), ("setup",), [self.root / f"baseline-{label}.json"])\n            self.add("score-" + label, lambda model=model: self.score(model), ("download-" + label,), [self.results_dir(model) / "summary.md"])\n        scores = ["score-" + tag for tag in BENCH_MODELS] + ["score-Qwen3.8-27B", "score-Gemma-4-E4B"]\n        self.add("compare-open-models", lambda: self.compare(False), scores, [art / "open_models_compare.md"])\n        self.add("compare-jev", lambda: self.compare(True), scores + ["score-Jev"], [art / "jev_compare.md"])\n        self.add("latency", lambda: self.python("scripts/bench_latency.py", "--backbone", STUDENT, "--ks", "1", "2", "4", "8", "16", "--repeats", "5", "--device", "cuda"),\n                 ("tests",), [art / "bench_latency.json"])\n        self.add("async", lambda: self.python("scripts/bench_async.py", "--backbone", STUDENT, "--rates", "5", "20", "50", "--n", "200", "--device", "cuda"),\n                 ("tests",), [art / "bench_async.json"])\n\n    def bundle(self, final=False):\n        """Logs/results always survive a Python exception. Runtime loss needs download.\n\n        Does not copy HF tokens, environment variables, .git, raw Jev databases,\n        cached model weights, or multi-GB extracted feature tensors.\n        """\n        paths = []\n        for pattern in ["*.json", "*.txt", "*.xml", "*.py", "logs/*.log", "coverage/*.json"]:\n            paths.extend(self.root.glob(pattern))\n        if self.repo.exists():\n            for pattern in ["artifacts/**/*.json", "artifacts/*.md", "runs/**/*.jsonl"]:\n                paths.extend(self.repo.glob(pattern))\n        if self.harness.exists():\n            for pattern in ["results/**/*.json", "results/**/*.md", "responses/LICENSE_RESPONSES.*", "LICENSE"]:\n                paths.extend(self.harness.glob(pattern))\n        paths = sorted({p for p in paths if p.is_file() and p.name != "bundle_manifest.json"})\n        manifest = [{"path": str(p.relative_to(self.root)), "bytes": p.stat().st_size, "sha256": sha256(p)} for p in paths]\n        write_json(self.root / "bundle_manifest.json", manifest)\n        paths = sorted(set(paths + [self.root / "bundle_manifest.json"]))\n        target = self.root / "results.zip"\n        temporary = target.with_suffix(".tmp")\n        with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED) as z:\n            for path in paths:\n                z.write(path, path.relative_to(self.root))\n        temporary.replace(target)\n        if final:\n            checkpoints = []\n            for name in list(HEADS) + ["lora"]:\n                directory = self.repo / "artifacts" / name\n                if directory.exists():\n                    checkpoints.extend(p for p in directory.iterdir() if p.is_file())\n            # Exact source records permit auditing upstream dataset drift.\n            records = self.repo / "artifacts" / "feats" / "records.pt"\n            if records.exists():\n                checkpoints.append(records)\n            with zipfile.ZipFile(self.root / "checkpoints.zip", "w", zipfile.ZIP_DEFLATED) as z:\n                for path in sorted(checkpoints):\n                    z.write(path, path.relative_to(self.root))\n\n    def run(self):\n        self.make_plan()\n        write_json(self.root / "stage_plan.json", [{"name": n, "dependencies": d, "outputs": [str(p.relative_to(self.root)) for p in o]} for n, _, d, o, _ in self.plan])\n        try:\n            for entry in self.plan:\n                self.stage(*entry)\n        finally:\n            self.bundle(final=True)\n        failed = [n for n, *_ in self.plan if self.status.get(n, {}).get("status") != "succeeded"]\n        self.emit("\\nRESULTS: " + str(self.root / "results.zip"))\n        self.emit("CHECKPOINTS: " + str(self.root / "checkpoints.zip"))\n        self.emit("ALL STAGES SUCCEEDED" if not failed else "INCOMPLETE: " + ", ".join(failed))\n        return 1 if failed else 0\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--root", default="/content/jex-full-run")\n    parser.add_argument("--jev-license-ack", action="store_true", help="Enable Jev response download after license review. Does not waive license restrictions.")\n    parser.add_argument("--allow-colab-optional-package-cleanup", action="store_true",\n                        help="In a verified hosted Colab runtime only, remove unused TorchAO older than 0.16.0 if it conflicts with pinned PEFT")\n    parser.add_argument("--plan-only", action="store_true", help="Print scope only; no cloning, installation, models, or experiments")\n    args = parser.parse_args()\n    if args.plan_only:\n        print(json.dumps(CONFIG, indent=2))\n        return 0\n    return Runner(args.root, jev_license_ack=args.jev_license_ack,\n                  allow_colab_optional_package_cleanup=args.allow_colab_optional_package_cleanup).run()\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
RUNNER.write_text(RUNNER_SOURCE)
assert hashlib.sha256(RUNNER.read_bytes()).hexdigest() == '7a5f719915c4ca67b8b552bd1cb108c091bbc54c6286d44292cd6c7af24c284e'
print("Prepared", RUNNER, "at", hashlib.sha256(RUNNER.read_bytes()).hexdigest())


In [ ]:
import signal, subprocess, sys
command = [sys.executable, "-u", str(RUNNER), "--root", str(RUN_ROOT)]
if JEV_RESPONSE_LICENSE_ACK:
    command.append("--jev-license-ack")
if COLAB_OPTIONAL_PACKAGE_CLEANUP:
    command.append("--allow-colab-optional-package-cleanup")
# Explicitly stream the pipe: inherited subprocess stdout may be invisible in Colab.
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, encoding="utf-8", errors="replace", bufsize=1)
try:
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
except KeyboardInterrupt:
    if process.poll() is None:
        process.send_signal(signal.SIGINT)
        print("Interrupt sent to this runner; waiting for logs and archives to finish.", flush=True)
        process.wait()
    raise
finally:
    process.stdout.close()
print("Runner exit code:", return_code)
if return_code:
    print("INCOMPLETE. Check stage_status.json and logs; successful independent stages are retained.")
else:
    print("All declared stages succeeded. Inspect coverage before interpreting benchmark means.")


In [ ]:
import json
from google.colab import files
status_path = RUN_ROOT / "stage_status.json"
if status_path.exists():
    status = json.loads(status_path.read_text())
    for stage, item in status.items():
        print(stage, item["status"], item.get("error", item.get("reason", "")))
for name in ["results.zip", "checkpoints.zip"]:
    path = RUN_ROOT / name
    if path.exists():
        print("Downloading", path.name, f"({path.stat().st_size / 2**20:.1f} MiB)")
        files.download(str(path))
    else:
        print("Unavailable:", path)
